In [ ]:
import os
os.environ["XFORMERS_DISABLED"] = "1"
WORK = "/lustre/fswork/projects/rech/rbw/ucw75ke"
PROJECT_ROOT = f"{WORK}/projects/GradientDistillation"
import gc
import sys
from dataclasses import dataclass
from typing import Callable
 
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
from tqdm.notebook import tqdm

import sys
p = f"{WORK}/python_user/h100/lib/python3.12/site-packages"
if p not in sys.path:
    sys.path.append(p)

import kmedoids

In [ ]:


 
os.environ["HOME"] = WORK
os.environ["TORCH_HOME"] = f"{WORK}/.cache/torch"
os.environ["HF_HOME"] = f"{WORK}/.cache/huggingface"
os.makedirs(os.environ["TORCH_HOME"], exist_ok=True)
 
for _p in (f"{PROJECT_ROOT}/src", PROJECT_ROOT):
    if _p not in sys.path:
        sys.path.insert(0, _p)
 
DATA_ROOT = f"{WORK}/datasets/aqua20/data/aqua20"
NUM_CLASSES = 20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESOLUTION = 252
 
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
CLIP_MEAN, CLIP_STD = (0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)
 
ARCHS = {
    "dinov2_vitb": dict(res=RESOLUTION, mean=IMAGENET_MEAN, std=IMAGENET_STD),
    "clip_vitb":   dict(res=224,        mean=CLIP_MEAN,     std=CLIP_STD),
    "mocov3_vitb": dict(res=224,        mean=IMAGENET_MEAN, std=IMAGENET_STD),
}

In [ ]:
IPCS = (1, 3, 5)
SEL_MODES = ("random", "medoids")
SEL_SEEDS = (3407, 42, 1234, 2024, 7)      # mirrors the distillation seeds
PROBE_SEEDS = (0, 1, 2, 3, 4)
PROBE_KW = dict(max_steps=5000, lr=1e-3, eval_every=1000, standardize_feats="scalar")

In [ ]:
SEL_SPACE = "fixed"
SEL_BACKBONE = "dinov2_vitb"
 
INCLUDE_FULL = True   # full-data reference, same probe protocol (nearly free)
 
OUT_RAW = "real_coreset_raw.csv"
OUT_SUMMARY = "real_coreset_summary.csv"

In [ ]:
@dataclass
class Backbone:
    name: str
    model: nn.Module
    forward: Callable[[torch.Tensor], torch.Tensor]
    mean: tuple
    std: tuple
    res: int
    feat_dim: int
 
 
def load_backbone(name: str) -> Backbone:
    cfg = ARCHS[name]
    if name == "dinov2_vitb":
        m = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")
        fwd = lambda x: m(x)                                    # CLS, 768
    elif name == "clip_vitb":
        import clip
        m = clip.load("ViT-B/32")[0].visual.float()
        fwd = lambda x: m(x)                                    # 512
    elif name == "mocov3_vitb":
        from src.models.moco_vision_tansformer import VisionTransformerMoCoV3
        m = VisionTransformerMoCoV3.from_pretrained("nyu-visionx/moco-v3-vit-b", num_classes=0)
        fwd = lambda x: m(x)                                    # 768
    else:
        raise ValueError(name)
 
    m.eval().requires_grad_(False).to(DEVICE)
    with torch.no_grad():
        feat_dim = fwd(torch.zeros(1, 3, cfg["res"], cfg["res"], device=DEVICE)).shape[-1]
    return Backbone(name, m, fwd, cfg["mean"], cfg["std"], cfg["res"], feat_dim)
 
 
def make_transform(bb: Backbone):
    return transforms.Compose([
        transforms.Resize(bb.res),
        transforms.CenterCrop(bb.res),
        transforms.ToTensor(),
        transforms.Normalize(bb.mean, bb.std),
    ])
 
 
@torch.no_grad()
def extract_features(loader, bb: Backbone, desc="Extracting features"):
    feats, labels = [], []
    for x, y in tqdm(loader, desc=desc, leave=False):
        feats.append(bb.forward(x.to(DEVICE)).float().cpu())
        labels.append(y)
    return torch.cat(feats), torch.cat(labels)
 

In [ ]:
def standardize(train_feats, test_feats, mode="scalar"):
    """Per-dimension centring plus rescaling.
 
    mode='scalar' : one global scale. Fixes the inter-backbone scale disparity
        (MoCoV3 features have much smaller norm) without estimating d standard
        deviations from n << d, which would amplify noise at IPC 1.
    mode='perdim' : classic standardisation. Robustness ablation only: the
        estimation bias depends on IPC.
    Stats computed on train only (= the coreset).
    """
    mu = train_feats.mean(0, keepdim=True)
    tr, te = train_feats - mu, test_feats - mu
    if mode == "none":
        return train_feats, test_feats
    if mode == "scalar":
        s = tr.std().clamp_min(1e-6)
    elif mode == "perdim":
        s = tr.std(0, keepdim=True).clamp_min(1e-6)
    else:
        raise ValueError(f"unknown mode: {mode!r}")
    return tr / s, te / s
 
 
def _infinite(loader):
    """Infinite batch stream WITH reshuffling on every pass.
 
    itertools.cycle would memorise the first pass and replay it identically,
    cancelling the DataLoader's shuffle.
    """
    while True:
        yield from loader
 
 
def train_linear_probe(train_feats, train_labels, test_feats, test_labels,
                       feat_dim=768, max_steps=2000, lr=1e-3,
                       batch_size=256, eval_every=500, seed=0,
                       standardize_feats="scalar", verbose=False):
    if standardize_feats != "none":
        train_feats, test_feats = standardize(train_feats, test_feats, mode=standardize_feats)
 
    torch.manual_seed(seed)
    np.random.seed(seed)
 
    head = nn.Linear(feat_dim, NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
 
    g = torch.Generator()
    g.manual_seed(seed)
    bs = min(batch_size, len(train_labels))
    loader = DataLoader(TensorDataset(train_feats, train_labels),
                        batch_size=bs, shuffle=True, generator=g, drop_last=False)
    test_loader = DataLoader(TensorDataset(test_feats, test_labels),
                             batch_size=512, shuffle=False)
 
    @torch.no_grad()
    def evaluate():
        head.eval()
        preds, ys = [], []
        for feats, y in test_loader:
            preds.append(head(feats.to(DEVICE)).argmax(1).cpu())
            ys.append(y)
        preds, ys = torch.cat(preds).numpy(), torch.cat(ys).numpy()
        head.train()
        return (f1_score(ys, preds, average="macro"),
                f1_score(ys, preds, average="weighted"))
 
    history = {}
    stream = _infinite(loader)
    run_loss, run_correct, run_n = 0.0, 0, 0
 
    for step in tqdm(range(1, max_steps + 1), desc="Probe", leave=False):
        feats, y = next(stream)
        feats, y = feats.to(DEVICE), y.to(DEVICE)
        logits = head(feats)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
 
        run_loss += loss.item() * len(y)
        run_correct += (logits.argmax(1) == y).sum().item()
        run_n += len(y)
 
        if step % eval_every == 0 or step == max_steps:
            f1m, f1w = evaluate()
            history[step] = {"loss": run_loss / run_n,
                             "train_acc": run_correct / run_n,
                             "f1_macro": f1m, "f1_weighted": f1w}
            if verbose:
                h = history[step]
                tqdm.write(f"step {step:>5}/{max_steps} | loss {h['loss']:.4f} | "
                           f"train acc {h['train_acc']*100:.1f}% | "
                           f"F1 macro {h['f1_macro']*100:.1f}%")
            run_loss, run_correct, run_n = 0.0, 0, 0
 
    # guard: the protocol assumes the probe has converged
    ckpts = sorted(history)
    if len(ckpts) >= 2:
        drift = abs(history[ckpts[-1]]["f1_macro"] - history[ckpts[-2]]["f1_macro"]) * 100
        if drift > 0.5:
            print(f"[warn] probe not converged: dF1={drift:.2f} pt between step "
                  f"{ckpts[-2]} and {ckpts[-1]} (seed={seed}) -> raise max_steps")
 
    return head, history
 
 
def final_metrics(history: dict) -> dict:
    """Last checkpoint. Never the max over the test set."""
    last = max(history)
    return {k: float(history[last][k]) for k in ("f1_macro", "f1_weighted", "train_acc")}
 
 

In [ ]:

def select_random(labels_np: np.ndarray, ipc: int, seed: int,
                  num_classes: int = NUM_CLASSES) -> dict:
    """IPC uniformly drawn indices per class."""
    rng = np.random.default_rng(seed)
    out = {}
    for c in range(num_classes):
        cls_idx = np.where(labels_np == c)[0]
        k = min(ipc, len(cls_idx))
        out[c] = rng.choice(cls_idx, size=k, replace=False).tolist()
    return out
 
 
def select_medoids(features: torch.Tensor, labels_np: np.ndarray, ipc: int, seed: int,
                   num_classes: int = NUM_CLASSES) -> dict:
    """k-medoids per class on cosine distance, mirroring MedoidInitializer."""
    feats_np = F.normalize(features, dim=1).numpy().astype(np.float32)
    out = {}
    for c in range(num_classes):
        cls_idx = np.where(labels_np == c)[0]
        Fc = feats_np[cls_idx]
        k = min(ipc, len(cls_idx))
        D = np.clip(1.0 - Fc @ Fc.T, 0.0, 2.0)
        np.fill_diagonal(D, 0.0)
        res = kmedoids.fasterpam(D, k, random_state=seed)
        out[c] = cls_idx[res.medoids].tolist()
    return out
 
 
def flatten(sel: dict) -> np.ndarray:
    """{class: [idx, ...]} -> flat sorted index array."""
    return np.array(sorted(i for v in sel.values() for i in v), dtype=np.int64)
 

In [ ]:
sel_bb = load_backbone(SEL_BACKBONE)
sel_ds = datasets.ImageFolder(f"{DATA_ROOT}/train", transform=make_transform(sel_bb))
 
sel_feats, sel_labels = extract_features(
    DataLoader(sel_ds, batch_size=64, shuffle=False, num_workers=8),
    sel_bb, f"[{SEL_BACKBONE}] selection space")
 
sel_labels_np = sel_labels.numpy()
counts = np.bincount(sel_labels_np, minlength=NUM_CLASSES)
print(f"train: {len(sel_labels_np)} images | class counts "
      f"min={counts.min()} max={counts.max()} ratio={counts.max()/counts.min():.1f}:1")
assert counts.min() >= max(IPCS), f"a class has fewer than {max(IPCS)} images"
 
del sel_bb
gc.collect()
torch.cuda.empty_cache()
 

In [ ]:
selections = {}
for ipc in IPCS:
    for seed in SEL_SEEDS:
        selections[("random", ipc, seed)] = select_random(sel_labels_np, ipc, seed)
        selections[("medoids", ipc, seed)] = select_medoids(sel_feats, sel_labels_np, ipc, seed)
 
for ipc in IPCS:
    for mode in SEL_MODES:
        uniq = {tuple(flatten(selections[(mode, ipc, s)])) for s in SEL_SEEDS}
        print(f"ipc={ipc} {mode:8s}: {len(uniq)}/{len(SEL_SEEDS)} distinct index sets")
 
if SEL_SPACE == "fixed":
    del sel_feats
    gc.collect()

In [ ]:
rows = []
 
for arch in ARCHS:
    bb = load_backbone(arch)
    tf = make_transform(bb)
 
    test_feats, test_labels = extract_features(
        DataLoader(datasets.ImageFolder(f"{DATA_ROOT}/test", transform=tf),
                   batch_size=64, shuffle=False, num_workers=8),
        bb, f"[{arch}] test")
 
    train_feats, train_labels = extract_features(
        DataLoader(datasets.ImageFolder(f"{DATA_ROOT}/train", transform=tf),
                   batch_size=64, shuffle=False, num_workers=8),
        bb, f"[{arch}] full train")
 
    assert torch.equal(train_labels, sel_labels), \
        "train ordering differs from the selection space; indices are not transferable"
 
    if INCLUDE_FULL:
        for ps in PROBE_SEEDS:
            _, hist = train_linear_probe(train_feats, train_labels, test_feats, test_labels,
                                         feat_dim=bb.feat_dim, seed=ps, **PROBE_KW)
            m = final_metrics(hist)
            rows.append({"arch": arch, "IPC": None, "Variant": "full",
                         "dseed": None, "pseed": ps,
                         "F1 macro (%)": m["f1_macro"] * 100,
                         "F1 weighted (%)": m["f1_weighted"] * 100})
 
    # native-space medoids: recompute in this backbone's own feature space
    if SEL_SPACE == "native":
        for ipc in IPCS:
            for seed in SEL_SEEDS:
                selections[("medoids", ipc, seed)] = select_medoids(
                    train_feats, sel_labels_np, ipc, seed)
 
    for (mode, ipc, dseed), sel in selections.items():
        idx = torch.from_numpy(flatten(sel))
        sub_feats, sub_labels = train_feats[idx], train_labels[idx]
 
        for ps in PROBE_SEEDS:
            _, hist = train_linear_probe(
                sub_feats, sub_labels, test_feats, test_labels,
                feat_dim=bb.feat_dim, seed=ps, **PROBE_KW)
            m = final_metrics(hist)
            rows.append({"arch": arch, "IPC": ipc, "Variant": f"real_{mode}",
                         "dseed": dseed, "pseed": ps,
                         "F1 macro (%)": m["f1_macro"] * 100,
                         "F1 weighted (%)": m["f1_weighted"] * 100})
 
    del bb, tf, test_feats, test_labels, train_feats, train_labels
    gc.collect()
    torch.cuda.empty_cache()
 
df_real = pd.DataFrame(rows)
df_real.to_csv(OUT_RAW, index=False)
print(f"{len(df_real)} raw measurements -> {OUT_RAW}")
 

In [ ]:
per_run = (df_real.groupby(["arch", "IPC", "Variant", "dseed"], dropna=False, observed=True)
                  .agg(macro=("F1 macro (%)", "mean"),
                       weighted=("F1 weighted (%)", "mean"))
                  .reset_index())
 
summary = (per_run.groupby(["arch", "IPC", "Variant"], dropna=False, observed=True)
                  .agg(macro_mean=("macro", "mean"), macro_std=("macro", "std"),
                       weighted_mean=("weighted", "mean"), weighted_std=("weighted", "std"),
                       n=("macro", "count"))
                  .reset_index()
                  .sort_values(["arch", "IPC", "Variant"], na_position="last")
                  .reset_index(drop=True))
 
fmt = lambda mu, sd: f"{mu:.2f}" if pd.isna(sd) else f"{mu:.2f} ± {sd:.2f}"
summary["F1 macro"] = summary.apply(lambda r: fmt(r.macro_mean, r.macro_std), axis=1)
summary["F1 weighted"] = summary.apply(lambda r: fmt(r.weighted_mean, r.weighted_std), axis=1)
 
print(summary[["arch", "IPC", "Variant", "F1 macro", "F1 weighted", "n"]].to_string(index=False))
summary.to_csv(OUT_SUMMARY, index=False)
 
 
# %% [markdown]
# ## Paired medoids − random contrast
#
# Same selection seed on both sides, so this is paired. Sign consistency across
# seeds is more informative than the mean gap: with n=5 a t-test has little power,
# whereas 5/5 same-sign is already suggestive.
 
# %%
deltas = (per_run[per_run["Variant"] != "full"]
          .pivot_table(index=["arch", "IPC", "dseed"], columns="Variant", values="macro")
          .reset_index())
 
if {"real_medoids", "real_random"} <= set(deltas.columns):
    deltas["delta"] = deltas["real_medoids"] - deltas["real_random"]
    contrast = (deltas.groupby(["arch", "IPC"])["delta"]
                      .agg(mean="mean", std="std",
                           pos=lambda s: int((s > 0).sum()), n="count")
                      .reset_index())
    contrast["sign"] = contrast.apply(lambda r: f"{r.pos}/{r.n}", axis=1)
    print("medoids - random, macro F1 (pt):")
    print(contrast[["arch", "IPC", "mean", "std", "sign"]].round(2).to_string(index=False))
 